# 27 — Interview Questions: FAANG / Unicorn Level

50 production-level questions with exact answers. These are the actual Pandas questions
asked at Amazon, Google, Flipkart, Swiggy, Zepto, FAANG, and DS interviews.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 300

emp = pd.DataFrame({
    'emp_id':     range(1001, 1001+n),
    'name':       [f'Emp_{i:03d}' for i in range(n)],
    'dept':       np.random.choice(['Eng', 'Sales', 'HR', 'Mkt', 'Finance'], n),
    'level':      np.random.choice(['Junior', 'Mid', 'Senior', 'Lead'], n),
    'salary':     np.random.randint(40000, 200000, n).astype(float),
    'bonus':      np.random.randint(0, 40000, n).astype(float),
    'rating':     np.random.choice([1.0, 2.0, 3.0, 4.0, 5.0], n, p=[0.05, 0.1, 0.25, 0.4, 0.2]),
    'join_year':  np.random.randint(2015, 2024, n),
    'city':       np.random.choice(['Bangalore', 'Mumbai', 'Delhi', 'Pune', 'Chennai'], n),
    'is_remote':  np.random.choice([0, 1], n, p=[0.6, 0.4]),
})

# Introduce NaN
emp.loc[np.random.choice(n, 20, replace=False), 'salary'] = np.nan
emp.loc[np.random.choice(n, 15, replace=False), 'bonus']  = np.nan

print(f"Dataset: {emp.shape}")

---
### Q1: Find 2nd highest salary per department

In [ ]:
# Method 1: groupby + apply
second_highest = (
    emp.dropna(subset=['salary'])
    .groupby('dept')['salary']
    .apply(lambda x: x.nlargest(2).iloc[-1] if len(x) >= 2 else np.nan)
)
print("2nd highest salary per dept:")
print(second_highest.round(0))

# Method 2: rank-based
second_highest_rank = (
    emp.dropna(subset=['salary'])
    .assign(rank=lambda df: df.groupby('dept')['salary'].rank(ascending=False, method='dense'))
    .query('rank == 2')
    .groupby('dept')['salary'].min().round(0)
)
print("\nRank-based:")
print(second_highest_rank)

---
### Q2: Find employees who earn more than their department's average

In [ ]:
emp['dept_avg'] = emp.groupby('dept')['salary'].transform('mean')
above_avg = emp[emp['salary'] > emp['dept_avg']]
print(f"Above dept average: {len(above_avg)} / {emp['salary'].notna().sum()}")
print(above_avg[['name', 'dept', 'salary', 'dept_avg']].head(5))

---
### Q3: Find the department with the highest average salary-to-bonus ratio

In [ ]:
ratio = (
    emp.dropna(subset=['salary', 'bonus'])
    .groupby('dept')
    .apply(lambda g: (g['bonus'] / g['salary']).mean())
    .round(4)
    .sort_values(ascending=False)
)
print("Avg bonus/salary ratio by dept:")
print(ratio)

---
### Q4: Detect and remove outliers using IQR

In [ ]:
def remove_outliers_iqr(df, col):
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return df[(df[col] >= lower) & (df[col] <= upper)]

clean_salary = remove_outliers_iqr(emp.dropna(subset=['salary']), 'salary')
print(f"Before: {emp['salary'].notna().sum()}  After IQR filter: {len(clean_salary)}")

---
### Q5: Find all employees hired in the last 3 years

In [ ]:
current_year = 2024
recent_hires = emp[emp['join_year'] >= current_year - 3]
print(f"Hired in last 3 years: {len(recent_hires)}")
print(recent_hires['join_year'].value_counts().sort_index())

---
### Q6: What percentage of total payroll does each department consume?

In [ ]:
total_payroll = emp['salary'].sum()
dept_share = (
    emp.groupby('dept')['salary'].sum()
    .div(total_payroll)
    .mul(100)
    .round(1)
    .sort_values(ascending=False)
)
print(dept_share)

---
### Q7: Find employees whose salary increased > 20% from their level median

In [ ]:
emp['level_median'] = emp.groupby('level')['salary'].transform('median')
high_earners = emp[
    emp['salary'] > emp['level_median'] * 1.20
]
print(f"Earning >20% above level median: {len(high_earners)}")
print(high_earners[['name', 'level', 'salary', 'level_median']].head(5))

---
### Q8: Get all duplicate rows but keep only first occurrence

In [ ]:
# Show duplicates (any row that matches another on emp_id)
dup_data = pd.concat([
    emp.head(5),
    emp.head(3)  # manually introduce duplication
], ignore_index=True)

print(f"Total rows: {len(dup_data)}")
print(f"Duplicated rows: {dup_data.duplicated().sum()}")

deduped = dup_data.drop_duplicates()
print(f"After drop_duplicates: {len(deduped)}")

---
### Q9: Compute running headcount by department sorted by join_year

In [ ]:
by_dept_year = (
    emp.groupby(['dept', 'join_year'])
    .size()
    .reset_index(name='hires')
    .sort_values(['dept', 'join_year'])
)
by_dept_year['running_headcount'] = by_dept_year.groupby('dept')['hires'].cumsum()
print(by_dept_year)

---
### Q10: Find employees with salary in the top 10% overall

In [ ]:
p90 = emp['salary'].quantile(0.9)
top10pct = emp[emp['salary'] > p90]
print(f"90th percentile salary: ₹{p90:,.0f}")
print(f"Top 10% earners: {len(top10pct)}")
print(top10pct[['name', 'dept', 'salary']].sort_values('salary', ascending=False).head(10))

---
### Q11: Pivot — avg salary by dept (rows) × level (columns)

In [ ]:
pt = pd.pivot_table(emp, values='salary', index='dept', columns='level', aggfunc='mean').round(0)
print(pt)

---
### Q12: Which city has the highest average bonus?

In [ ]:
city_bonus = emp.groupby('city')['bonus'].mean().round(0).sort_values(ascending=False)
print(city_bonus)
print(f"\nCity with highest avg bonus: {city_bonus.idxmax()} (₹{city_bonus.max():,.0f})")

---
### Q13: For each department, find the employee with the longest tenure

In [ ]:
emp['tenure'] = 2024 - emp['join_year']

longest_tenure = (
    emp
    .sort_values('tenure', ascending=False)
    .drop_duplicates(subset='dept', keep='first')
    [['dept', 'name', 'tenure', 'salary']]
    .sort_values('dept')
)
print(longest_tenure)

---
### Q14: Find employees who are the only person at their level in their department

In [ ]:
emp['dept_level_count'] = emp.groupby(['dept', 'level'])['emp_id'].transform('count')
sole_members = emp[emp['dept_level_count'] == 1]
print(f"Sole member at dept×level: {len(sole_members)}")
print(sole_members[['name', 'dept', 'level', 'salary']].head(8))

---
### Q15: Compute median absolute deviation (MAD) as a robust outlier score

In [ ]:
salary_valid = emp.dropna(subset=['salary'])
median = salary_valid['salary'].median()
mad = (salary_valid['salary'] - median).abs().median()

salary_valid = salary_valid.copy()
salary_valid['mad_score'] = ((salary_valid['salary'] - median) / mad).round(2)

print(f"Median: {median:,.0f}  MAD: {mad:,.0f}")
print("\nHighest MAD scores (likely outliers):")
print(salary_valid.nlargest(5, 'mad_score')[['name', 'dept', 'salary', 'mad_score']])

---
### Q16: Multiple self-joins — find pairs with same dept AND same city

In [ ]:
pairs = pd.merge(
    emp[['emp_id', 'name', 'dept', 'city']],
    emp[['emp_id', 'name', 'dept', 'city']],
    on=['dept', 'city'],
    suffixes=('_a', '_b')
)
pairs = pairs[pairs['emp_id_a'] < pairs['emp_id_b']]
print(f"Pairs (same dept + city): {len(pairs):,}")
print(pairs[['name_a', 'name_b', 'dept', 'city']].head(5))

---
### Q17: What fraction of remote employees are in each department?

In [ ]:
remote_pct = emp.groupby('dept')['is_remote'].mean().mul(100).round(1).sort_values(ascending=False)
print("Remote employee % by department:")
print(remote_pct)

---
### Q18: Month-over-month salary budget change (simulate time series)

In [ ]:
monthly_budget = pd.Series(
    np.random.randint(10_000_000, 20_000_000, 24),
    index=pd.date_range('2022-01', periods=24, freq='ME')
)

mom_change = monthly_budget.pct_change().mul(100).round(2)
print(pd.DataFrame({'budget': monthly_budget, 'mom_pct': mom_change}).head(10))

---
### Q19: Find all employees whose salary increased from year to year (cummax pattern)

In [ ]:
# Simulate annual salary records per employee
annual = pd.DataFrame({
    'emp_id': np.repeat([1001, 1002, 1003], 4),
    'year':   [2020, 2021, 2022, 2023] * 3,
    'salary': [50000, 55000, 53000, 60000,   # 1001: dip in 2022
               70000, 75000, 80000, 85000,   # 1002: steady increase
               90000, 88000, 91000, 95000]   # 1003: dip in 2021
})

annual['cummax_salary'] = annual.groupby('emp_id')['salary'].cummax()
annual['is_new_high']   = annual['salary'] == annual['cummax_salary']
print(annual)

---
### Q20: Anti-join — which departments have 0 Leads?

In [ ]:
all_depts   = pd.DataFrame({'dept': emp['dept'].unique()})
lead_depts  = pd.DataFrame({'dept': emp[emp['level'] == 'Lead']['dept'].unique()})

no_leads = (
    pd.merge(all_depts, lead_depts, on='dept', how='left', indicator=True)
    .query('_merge == "left_only"')
    .drop(columns='_merge')
)
print("Departments with no Lead employees:")
print(no_leads)

---
### Q21: How to safely use eval() for memory-efficient column computation?

In [ ]:
emp_copy = emp.dropna(subset=['salary', 'bonus']).copy()

# Using eval() — avoids creating intermediate arrays
emp_copy.eval('total_comp = salary + bonus', inplace=True)
print(emp_copy[['name', 'salary', 'bonus', 'total_comp']].head(5))

---
### Q22: Explain Copy-on-Write (CoW) — the most important Pandas 2.0 change

In [ ]:
# Pre-Pandas 2.0: chained assignment silently fails or warns
# Pandas 2.0+: CoW means ALL modifications require assignment to a new variable

# WRONG (may not modify original in Pandas 2.0+):
# emp[emp['dept'] == 'Eng']['salary'] = 999

# CORRECT: use loc or assign
emp_cow = emp.copy()
emp_cow.loc[emp_cow['dept'] == 'Eng', 'salary'] = 999  # ✅ correct

print("After CoW-safe assignment:")
print(emp_cow[emp_cow['dept'] == 'Eng']['salary'].head(3).tolist())

# Restore
emp_cow = emp.copy()

---
### Q23: Find all rows where salary is NULL but bonus is not NULL

In [ ]:
result = emp[emp['salary'].isna() & emp['bonus'].notna()]
print(f"Salary NULL but bonus not NULL: {len(result)}")
print(result[['name', 'dept', 'salary', 'bonus']].head(5))

---
### Q24: np.select — multi-condition vectorized replacement

In [ ]:
conditions = [
    emp['rating'] >= 4.5,
    emp['rating'] >= 3.5,
    emp['rating'] >= 2.5,
]
choices = ['Excellent', 'Good', 'Average']

emp['performance'] = np.select(conditions, choices, default='Poor')
print(emp['performance'].value_counts())

---
### Q25: Build a salary band system using pd.cut + conditional logic

In [ ]:
emp['salary_band'] = pd.cut(
    emp['salary'],
    bins=[0, 60000, 100000, 140000, 200001],
    labels=['L1', 'L2', 'L3', 'L4']
)
print(emp['salary_band'].value_counts().sort_index())

# Avg bonus by salary band
print(emp.groupby('salary_band', observed=True)['bonus'].mean().round(0))